# Prétraitement des données

Notebook de préparation des données.

In [25]:
import pandas as pd


In [15]:
df_brut = pd.read_excel(
    "../data/raw/rendement_départementale.xlsx",
    sheet_name="Maïs",
    header=None
)

# Afficher les 15 premières lignes et 6 premières colonnes
print(df_brut.iloc[:15, :6].to_string())

                   0           1        2      3         4         5
0                NaN         NaN      NaN    NaN       NaN       NaN
1                NaN         NaN      NaN    NaN       NaN       NaN
2                NaN         NaN      NaN    NaN       NaN       NaN
3                NaN         NaN      NaN    NaN       NaN       NaN
4                NaN         NaN      NaN    NaN       NaN       NaN
5                NaN         NaN      NaN    NaN       NaN       NaN
6        Indicateurs       Zones    Units  Scale    1995.0    1996.0
7   Superficie en ha     Atakora  hectare    NaN   18509.0   20395.0
8   Superficie en ha       Bénin  hectare  units  493494.0  516539.0
9   Superficie en ha     Alibori  hectare    NaN   31080.0   33493.0
10  Superficie en ha  Atlantique  hectare  units   91324.0   89300.0
11  Superficie en ha      Borgou  hectare    NaN   64119.0   60840.0
12  Superficie en ha    Collines  hectare    NaN   39860.0   40796.0
13  Superficie en ha       Donga  

In [16]:
df_mais = pd.read_excel(
    "../data/raw/rendement_départementale.xlsx",
    sheet_name="Maïs",
    header=6
)

print(df_mais.shape)
print(df_mais["Indicateurs"].unique())
print(df_mais["Zones"].unique())

(39, 33)
<StringArray>
['Superficie en ha', 'Rendement en kg/ha', 'Production en tonne']
Length: 3, dtype: str
<StringArray>
[   'Atakora',      'Bénin',    'Alibori', 'Atlantique',     'Borgou',
   'Collines',      'Donga',     'Kouffo',   'Littoral',       'Mono',
      'Ouémé',        'Zou',    'Plateau']
Length: 13, dtype: str


In [17]:
# Garder uniquement les rendements départementaux (pas le Bénin entier)
departements_liste = [
    'Atakora', 'Alibori', 'Atlantique', 'Borgou', 'Collines',
    'Donga', 'Kouffo', 'Littoral', 'Mono', 'Ouémé', 'Zou', 'Plateau'
]

df_rendement = df_mais[
    (df_mais["Indicateurs"] == "Rendement en kg/ha") &
    (df_mais["Zones"].isin(departements_liste))
].copy()

print(df_rendement.shape)
print(df_rendement.head())

(12, 33)
           Indicateurs       Zones  Units  Scale         1995         1996  \
13  Rendement en kg/ha     Atakora  KG/HA    NaN  1366.200227  1252.905124   
15  Rendement en kg/ha     Alibori  KG/HA    NaN  1294.530245  1470.486370   
16  Rendement en kg/ha  Atlantique  KG/HA  units   967.248478  1003.493841   
17  Rendement en kg/ha      Borgou  KG/HA    NaN  1147.787707  1101.298488   
18  Rendement en kg/ha    Collines  KG/HA    NaN  1166.056197  1011.545250   

           1997         1998         1999         2000  ...    2014    2015  \
13  1043.074502  1343.334800  1447.957101  1431.334688  ...  2160.0  1874.0   
15  1503.272945  1488.096858  1579.730810  1535.098354  ...  1502.0  1287.0   
16  1096.098027   849.384956  1094.677891   947.251851  ...  1284.0   984.0   
17  1165.684239  1192.967518  1220.473578  1301.478061  ...  1803.0  1474.0   
18  1027.250149   975.485527   843.233354   953.770197  ...   942.0   986.0   

      2016    2017    2018    2019    2020    2

In [18]:
# Transformer en format long
annees = [col for col in df_rendement.columns if str(col).isdigit() or 
          (isinstance(col, float) and not pd.isna(col))]

# Les colonnes années sont des entiers ou floats
cols_annees = [c for c in df_rendement.columns if isinstance(c, (int, float)) 
               and not pd.isna(c)]

df_rendement_long = df_rendement.melt(
    id_vars=["Zones"],
    value_vars=cols_annees,
    var_name="annee",
    value_name="rendement_kg_ha"
)

# Nettoyer
df_rendement_long["annee"] = df_rendement_long["annee"].astype(int)
df_rendement_long = df_rendement_long.rename(columns={"Zones": "departement"})

# Filtrer 2003-2022
df_rendement_long = df_rendement_long[
    (df_rendement_long["annee"] >= 2003) &
    (df_rendement_long["annee"] <= 2022)
].copy()

# Convertir rendement en t/ha
df_rendement_long["rendement_t_ha"] = df_rendement_long["rendement_kg_ha"] / 1000

print(df_rendement_long.shape)
print(df_rendement_long.sort_values(["departement","annee"]).head(12))

(0, 4)
Empty DataFrame
Columns: [departement, annee, rendement_kg_ha, rendement_t_ha]
Index: []


In [19]:
print(type(df_rendement.columns[4]))
print(df_rendement.columns[4:].tolist())

<class 'str'>
['1995', '1996', '1997', '1998', '1999', '2000', '2001', '2002', '2003', '2004', '2005', '2006', '2007', '2008', '2009', '2010', '2011', '2012', '2013', '2014', '2015', '2016', '2017', '2018', '2019', '2020', '2021', '2022', '2023']


In [20]:
# Les colonnes années sont des strings
cols_annees = [c for c in df_rendement.columns if str(c).isdigit()]

df_rendement_long = df_rendement.melt(
    id_vars=["Zones"],
    value_vars=cols_annees,
    var_name="annee",
    value_name="rendement_kg_ha"
)

# Nettoyer
df_rendement_long["annee"] = df_rendement_long["annee"].astype(int)
df_rendement_long = df_rendement_long.rename(columns={"Zones": "departement"})

# Filtrer 2003-2022
df_rendement_long = df_rendement_long[
    (df_rendement_long["annee"] >= 2003) &
    (df_rendement_long["annee"] <= 2022)
].copy()

# Convertir en t/ha
df_rendement_long["rendement_t_ha"] = df_rendement_long["rendement_kg_ha"] / 1000

print(df_rendement_long.shape)
print(df_rendement_long.sort_values(["departement","annee"]).head(12))

(240, 4)
    departement  annee  rendement_kg_ha  rendement_t_ha
97      Alibori   2003      1467.597550        1.467598
109     Alibori   2004      1582.495733        1.582496
121     Alibori   2005      1363.330490        1.363330
133     Alibori   2006      1439.646522        1.439647
145     Alibori   2007      1315.106952        1.315107
157     Alibori   2008      1371.386668        1.371387
169     Alibori   2009      1412.126181        1.412126
181     Alibori   2010      1624.475358        1.624475
193     Alibori   2011      1703.026001        1.703026
205     Alibori   2012      1370.000000        1.370000
217     Alibori   2013      1615.000000        1.615000
229     Alibori   2014      1502.000000        1.502000


In [23]:
climat_annuel = pd.read_csv("../data/processed/climat_annuel.csv")
print("Climat    :", sorted(climat_annuel["departement"].unique()))

Climat    : ['Alibori', 'Atacora', 'Atlantique', 'Borgou', 'Collines', 'Couffo', 'Donga', 'Littoral', 'Mono', 'Oueme', 'Plateau', 'Zou']


In [24]:
df_rendement_long["departement"] = df_rendement_long["departement"].replace({
    "Kouffo": "Couffo",
    "Atakora": "Atacora"
})

print("Rendement corrigé :", sorted(df_rendement_long["departement"].unique()))
print("Climat            :", sorted(climat_annuel["departement"].unique()))

Rendement corrigé : ['Alibori', 'Atacora', 'Atlantique', 'Borgou', 'Collines', 'Couffo', 'Donga', 'Littoral', 'Mono', 'Ouémé', 'Plateau', 'Zou']
Climat            : ['Alibori', 'Atacora', 'Atlantique', 'Borgou', 'Collines', 'Couffo', 'Donga', 'Littoral', 'Mono', 'Oueme', 'Plateau', 'Zou']


In [26]:
climat_annuel["departement"] = climat_annuel["departement"].replace({
    "Oueme": "Ouémé"
})

print("Climat corrigé :", sorted(climat_annuel["departement"].unique()))

Climat corrigé : ['Alibori', 'Atacora', 'Atlantique', 'Borgou', 'Collines', 'Couffo', 'Donga', 'Littoral', 'Mono', 'Ouémé', 'Plateau', 'Zou']


In [27]:
# Fusion des données de rendement et climatiques
dataset_final = pd.merge(
    df_rendement_long,
    climat_annuel,
    on=["departement", "annee"],
    how="inner"
)

print(f"Shape final : {dataset_final.shape}")
print(f"Départements : {sorted(dataset_final['departement'].unique())}")
print(f"Années : {sorted(dataset_final['annee'].unique())}")
print(dataset_final.head())

# Sauvegarder
dataset_final.to_csv("../data/processed/dataset_final.csv", index=False)
print("\nSauvegardé dans data/processed/dataset_final.csv")

Shape final : (240, 83)
Départements : ['Alibori', 'Atacora', 'Atlantique', 'Borgou', 'Collines', 'Couffo', 'Donga', 'Littoral', 'Mono', 'Ouémé', 'Plateau', 'Zou']
Années : [np.int64(2003), np.int64(2004), np.int64(2005), np.int64(2006), np.int64(2007), np.int64(2008), np.int64(2009), np.int64(2010), np.int64(2011), np.int64(2012), np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022)]
  departement  annee  rendement_kg_ha  rendement_t_ha  precip_01_mm  \
0     Atacora   2003      1199.803831        1.199804          0.03   
1     Alibori   2003      1467.597550        1.467598          0.00   
2  Atlantique   2003      1000.416519        1.000417          1.45   
3      Borgou   2003      1264.762506        1.264763          0.06   
4    Collines   2003       933.707044        0.933707          0.94   

   temp_moy_01_c  temp_max_01_c  temp_min_01_c  humidite_01_pct  \
0          2

240 lignes, 83 colonnes, 12 départements, 20 années. Le dataset final est prêt.
Voilà ce qu'on a dans les 83 colonnes :

2 colonnes d'identification : departement, annee
2 colonnes cibles : rendement_kg_ha, rendement_t_ha
72 colonnes mensuelles : 6 paramètres × 12 mois
7 colonnes annuelles synthétiques